In [1]:
import pandas as pd
import os

In [2]:
def extract_placements(dir):
    files = os.listdir(dir)

    placements = {1: [], 2: [], 3: [], 4: [], "qfs": []}
    for file in files:
        df = pd.read_csv(os.path.join(dir, file))

        # Get the winner
        final_row = df.iloc[-1]
        assert final_row["stage"] == "Final"
        sim_winner  = final_row["winner"]
        sim_finalist = final_row["home"] if final_row["home"] != sim_winner else final_row["away"]
        placements[1].append(sim_winner)
        placements[2].append(sim_finalist)

        # Get the other semifinalists (third place match)
        third_place_match_row = df.iloc[-2]
        assert third_place_match_row["stage"] == "Third"
        sim_third = third_place_match_row["winner"]
        sim_fourth = third_place_match_row["home"] if third_place_match_row["home"] != sim_third else third_place_match_row["away"]
        placements[3].append(sim_third)
        placements[4].append(sim_fourth)

        # Get the other quarterfinalists
        quarterfinal_rows = df.iloc[-8:-4]
        sim_quarterfinalists = quarterfinal_rows["home"].tolist() + quarterfinal_rows["away"].tolist()
        sim_quarterfinalists = [q for q in sim_quarterfinalists if q not in [sim_winner,sim_finalist,sim_third,sim_fourth]]
        placements["qfs"].append(sim_quarterfinalists)

    return placements

In [3]:
def aggregate_stats(placements):
    # Winners
    winners = pd.Series(placements[1])
    TOTAL_SIMULATIONS = len(winners)
    print("Total simulations: ", TOTAL_SIMULATIONS)
    winners_prob = pd.DataFrame(winners.value_counts() / TOTAL_SIMULATIONS).rename(columns={"count": "win_prob"})

    # Semi-Finalists
    semifinalists = pd.Series(placements[1]+placements[2]+placements[3]+placements[4])
    assert len(semifinalists) == 4 * TOTAL_SIMULATIONS
    semifinalists_prob = pd.DataFrame(semifinalists.value_counts() / TOTAL_SIMULATIONS).rename(columns={"count": "sf_prob"})

    # Quarter-Finalists (flatten the qfs list of lists)
    quarterfinalists = pd.Series(placements[1]+placements[2]+placements[3]+placements[4]+[c for sim in placements["qfs"] for c in sim])
    assert len(quarterfinalists) == 8 * TOTAL_SIMULATIONS
    quarterfinalists_prob = pd.DataFrame(quarterfinalists.value_counts() / TOTAL_SIMULATIONS).rename(columns={"count": "qf_prob"})

    # Join together
    simulation_prob = quarterfinalists_prob.merge(
        semifinalists_prob, left_index=True, right_index=True, how="outer"
    ).merge(
        winners_prob, left_index=True, right_index=True, how="outer"
    )
    return simulation_prob.sort_values(by=["win_prob","sf_prob"], ascending=False)

# ELO-based simulations

In [4]:
# Load data
placements_elo = extract_placements("simulation_results/elo")

In [5]:
simulation_elo_prob = aggregate_stats(placements_elo)
simulation_elo_prob

Total simulations:  10000


,qf_prob,sf_prob,win_prob
Spain,0.7553,0.6936,0.4159
Argentina,0.7206,0.5814,0.2438
France,0.6845,0.5262,0.1307
England,0.5554,0.3609,0.0612
Portugal,0.4166,0.1809,0.0291
Brazil,0.4389,0.2480,0.0267
Colombia,0.3888,0.1662,0.0229
Netherlands,0.4321,0.1830,0.0149
Ecuador,0.2803,0.1434,0.0091
Germany,0.2683,0.1345,0.0085


# Odds-based simulations

In [6]:
# Load data
placements_odds = extract_placements("simulation_results/odds")

In [7]:
simulation_odds_prob = aggregate_stats(placements_odds)
simulation_odds_prob

Total simulations:  10000


,qf_prob,sf_prob,win_prob
Spain,0.6826,0.5699,0.2262
France,0.6812,0.5177,0.2061
England,0.7048,0.4568,0.1446
Portugal,0.6934,0.4449,0.1258
Argentina,0.6028,0.3777,0.0928
Brazil,0.5377,0.3030,0.0809
Germany,0.3718,0.2326,0.0443
Netherlands,0.4373,0.1807,0.0267
Norway,0.3069,0.1301,0.0114
Belgium,0.4475,0.1283,0.0108


# Combine simulations

In [ ]:
compare_simulation_prob = simulation_elo_prob.merge(
    simulation_odds_prob,
    left_index=True,
    right_index=True,
    suffixes=("_elo", "_odds"),
    how="outer"
)
compare_simulation_prob[["qf_prob_elo", "qf_prob_odds", "sf_prob_elo", "sf_prob_odds", "win_prob_elo", "win_prob_odds"]] = \
    compare_simulation_prob[["qf_prob_elo", "qf_prob_odds", "sf_prob_elo", "sf_prob_odds", "win_prob_elo", "win_prob_odds"]].fillna(0)

compare_simulation_prob["qf_prob_avg"] = compare_simulation_prob[["qf_prob_elo", "qf_prob_odds"]].mean(axis=1)
compare_simulation_prob["sf_prob_avg"] = compare_simulation_prob[["sf_prob_elo", "sf_prob_odds"]].mean(axis=1)  # LOOK AT THIS
compare_simulation_prob["win_prob_avg"] = compare_simulation_prob[["win_prob_elo", "win_prob_odds"]].mean(axis=1)

# A combination (average) of ELO-based and odds-based simulations ->
# -> taking the P(getting to the semifinals)/4, assuming it's a tossup who wins once you reach the SF.
compare_simulation_prob["win_prob_avg_any_of_the_semifinalists"] = compare_simulation_prob["sf_prob_avg"] / 4
# Turns out extremely similar to Opta predictions: https://theanalyst.com/articles/who-will-win-2026-fifa-world-cup-predictions-opta-supercomputer

compare_simulation_prob.sort_values(by=["win_prob_avg_any_of_the_semifinalists", "win_prob_avg", "sf_prob_avg"], ascending=False)

,qf_prob_elo,sf_prob_elo,win_prob_elo,qf_prob_odds,sf_prob_odds,win_prob_odds,qf_prob_avg,sf_prob_avg,win_prob_avg,win_prob_avg_any_of_the_semifinalists
Spain,0.7553,0.6936,0.4159,0.6826,0.5699,0.2262,0.71895,0.63175,0.32105,0.157938
France,0.6845,0.5262,0.1307,0.6812,0.5177,0.2061,0.68285,0.52195,0.16840,0.130488
Argentina,0.7206,0.5814,0.2438,0.6028,0.3777,0.0928,0.66170,0.47955,0.16830,0.119888
England,0.5554,0.3609,0.0612,0.7048,0.4568,0.1446,0.63010,0.40885,0.10290,0.102212
Portugal,0.4166,0.1809,0.0291,0.6934,0.4449,0.1258,0.55500,0.31290,0.07745,0.078225
Brazil,0.4389,0.2480,0.0267,0.5377,0.3030,0.0809,0.48830,0.27550,0.05380,0.068875
Germany,0.2683,0.1345,0.0085,0.3718,0.2326,0.0443,0.32005,0.18355,0.02640,0.045887
Netherlands,0.4321,0.1830,0.0149,0.4373,0.1807,0.0267,0.43470,0.18185,0.02080,0.045463
Colombia,0.3888,0.1662,0.0229,0.2559,0.1103,0.0061,0.32235,0.13825,0.01450,0.034562
Norway,0.2583,0.1070,0.0064,0.3069,0.1301,0.0114,0.28260,0.11855,0.00890,0.029637


In [11]:
compare_simulation_prob.shape

(47, 10)